# Independent EV/EBITDA reconciliation
Xinyu Zhang · xz3490 · STAT GR5398 · 12 October 2026

The hand calculation reads archived inputs and computes enterprise value, debt adjustment and per-share value independently. The engine is invoked afterward only for reconciliation. This is a forward-horizon arithmetic audit, not a present-value investment recommendation.

The engine uses the final estimate-column EBITDA, the mean of five historical EV/EBITDA multiples, net debt equal to 10% of enterprise value, and no discounting. The 2027E label is stale: the effective third horizon is FY2028 for AAPL/AMD/INTC and FY2029 for NVDA/CSCO. Current shares are inferred from market cap divided by price in the same archived quote. The report headline target of zero instead reflects a blocked FMP endpoint and is missing data.


In [1]:
from pathlib import Path
import importlib.util
import json
import sys
from urllib.parse import urlsplit
import numpy as np
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / "artifacts/baseline").is_dir()
TICKERS = ["NVDA", "AMD", "INTC", "AAPL", "CSCO"]

spec = importlib.util.spec_from_file_location("valuation_engine", ROOT / "artifacts/source_snapshot/modules/valuation_engine.py")
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)


In [2]:
results = []
inputs = {}
for ticker in TICKERS:
    run = ROOT / "artifacts/baseline" / ticker
    analysis = pd.read_csv(run / "analysis/financial_metrics_and_forecasts.csv", keep_default_na=False)
    key_metrics = pd.read_csv(run / "analysis/key_metrics_raw_data.csv")
    balance = pd.read_csv(run / "analysis/balance_sheet_raw_data.csv").sort_values("date").iloc[-1]
    for path in sorted(run.glob("render_*/audit/*_fmp_request.json")):
        request = json.loads(path.read_text())
        if urlsplit(request["url"]).path.endswith("/quote"):
            quote = json.loads(path.with_name(path.name.replace("_request", "_response")).read_text())["body"][0]
            break
    year = [c for c in analysis if c.endswith("E")][-1]
    EBITDA = float(analysis.loc[analysis.metrics == "EBITDA", year].iloc[0])
    revenue = float(analysis.loc[analysis.metrics == "Revenue", year].iloc[0])
    multiple = float(key_metrics.enterpriseValueOverEBITDA.dropna().mean())
    shares = float(quote["marketCap"]) / float(quote["price"])
    enterprise_value = EBITDA * multiple
    debt_proxy = 0.10 * enterprise_value
    hand_target = max((enterprise_value - debt_proxy) / shares, 0)
    alternative_target = max((enterprise_value - float(balance.netDebt)) / shares, 0)
    engine = module.ValuationEngine({"analysis": analysis, "key_metrics": key_metrics, "shares_outstanding": shares})
    engine_target = engine.calculate_ev_ebitda_valuation().target_price
    assert abs(hand_target - engine_target) < 1e-8
    inputs[ticker] = {"revenue": revenue, "margin": EBITDA / revenue, "multiple": multiple, "shares": shares, "base_price": hand_target}
    results.append({"ticker": ticker, "displayed_year": year, "multiple": multiple, "shares": shares,
                    "hand_price": hand_target, "engine_price": engine_target,
                    "difference": hand_target - engine_target, "balance_debt_alternative": alternative_target})
reconciliation = pd.DataFrame(results)
print(reconciliation.to_string(index=False, float_format=lambda x: f"{x:.8f}"))


ticker displayed_year    multiple               shares   hand_price  engine_price  difference  balance_debt_alternative
  NVDA          2027E 48.88485163 24147000000.00000000 334.73301251  334.73301251  0.00000000              371.89214916
   AMD          2027E 40.65038775  1630600000.00000000 377.83562493  377.83562493  0.00000000              420.47172139
  INTC          2027E 31.51950280  5043999746.99713135 105.28497177  105.28497177  0.00000000              110.57568864
  AAPL          2027E 23.00368080 14687356000.00000000 304.20343701  304.20343701  0.00000000              332.79913808
  CSCO          2027E 16.02531041  3941434664.99891758 124.45613528  124.45613528  0.00000000              132.62295074


The alternative uses FMP's archived balance-sheet net-debt field. It is a convention comparison: FMP net debt excludes short-term investments in these files, while the engine invents 10%-of-EV debt. It does not repair the historical-average multiple or the EBITDA definition change. Exact reconciliation therefore establishes reproducibility, not economic validity.

The engine's low/high prices use mean ± sample standard deviation of the multiples while holding the base debt proxy fixed. Its fallback 12x multiple and 3x standard deviation are not used because all five companies have historical multiples. No WACC or terminal growth applies to this EV/EBITDA method; no discount factor is applied to the forward EBITDA.


In [3]:
rows = []
for ticker, data in inputs.items():
    for revenue_delta in np.linspace(-0.05, 0.05, 5):
        for margin_delta in np.linspace(-0.02, 0.02, 5):
            EBITDA = data["revenue"] * (1 + revenue_delta) * (data["margin"] + margin_delta)
            price = max(0.90 * EBITDA * data["multiple"] / data["shares"], 0)
            rows.append({"ticker": ticker, "revenue_delta_pct": revenue_delta * 100,
                         "margin_delta_pp": margin_delta * 100, "target_price": price,
                         "price_change_pct": (price / data["base_price"] - 1) * 100})
sensitivity = pd.DataFrame(rows)
grid = sensitivity[sensitivity.ticker == "NVDA"].pivot(index="revenue_delta_pct", columns="margin_delta_pp", values="target_price")
print("NVDA price grid: rows are revenue %, columns are EBITDA margin percentage points")
print(grid.to_string(float_format=lambda x: f"{x:.2f}"))
print("All-company corner ranges (%)")
print(sensitivity.groupby("ticker").price_change_pct.agg(["min", "max"]).to_string(float_format=lambda x: f"{x:.3f}"))


NVDA price grid: rows are revenue %, columns are EBITDA margin percentage points
margin_delta_pp     -2.0   -1.0    0.0    1.0    2.0
revenue_delta_pct                                   
-5.0              309.34 313.67 318.00 322.32 326.65
-2.5              317.48 321.92 326.36 330.81 335.25
 0.0              325.62 330.18 334.73 339.29 343.84
 2.5              333.77 338.43 343.10 347.77 352.44
 5.0              341.91 346.69 351.47 356.25 361.03
All-company corner ranges (%)
           min    max
ticker               
AAPL    -9.241  9.688
AMD     -9.524 10.000
CSCO    -9.095  9.526
INTC   -11.209 11.863
NVDA    -7.585  7.857


This grid connects the same ±5% revenue-level shock and ±2 percentage-point margin ranges used by `sensitivity_analyzer.py` to the reproduced target-price formula. It uses the full-precision EBITDA/revenue ratio to avoid drift caused by one-decimal displayed margins. It is not a ±5 percentage-point shock compounded annually, a probabilistic confidence interval, or a full valuation-risk range; the multiple and shares remain fixed.

Primary filing checks: [NVIDIA FY2025 10-K](https://www.sec.gov/Archives/edgar/data/1045810/000104581025000023/nvda-20250126.htm) and [Apple FY2025 10-K](https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm). The archived raw-data and HTTP files in `artifacts/` identify all valuation inputs.
